In [1]:
# freezing the coding environment to these specific versions
!pip install langchain==0.3.23 langchain-huggingface langchain_community==0.3.21 faiss-cpu==1.10.0 akshare gradio --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 73.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are in

In [5]:
# 在 Google Colab 中新建一个代码单元格，粘贴以下代码并运行
# 注意：运行前请确保已挂载 Google Drive（如果需要读取之前的数据），或者让代码重新下载数据

import gradio as gr
import os
import pickle
import numpy as np
from langchain_community.document_loaders import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.vectorstores import FAISS

# ------------------------------
# 1. 配置 HF API Key
# ------------------------------
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "Your Hugging Face API Key"

# ------------------------------
# 2. 全局初始化：构建或加载 FAISS 索引 + 文本块列表
# ------------------------------
DATA_PATH = "data/AKSHARE_cjzc/stock_cjzc_em.csv"
INDEX_PATH = "data/cjzc_faiss_index"
CHUNKS_PKL = "data/cjzc_chunks.pkl"

def split_text(text):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    return splitter.split_text(text)

def initialize_rag():
    print("正在初始化 RAG 系统...")

    # 检查是否已有保存好的索引和文本块
    if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_PKL):
        print("发现已有 FAISS 索引和文本块缓存，直接加载...")
        with open(CHUNKS_PKL, "rb") as f:
            csv_chunks = pickle.load(f)
        embeddings = HuggingFaceEndpointEmbeddings(
            model="intfloat/multilingual-e5-large",
            task="feature-extraction",
            huggingfacehub_api_token=os.environ.get("HUGGINGFACEHUB_API_TOKEN"),
            )
        faiss_index = FAISS.load_local(INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
        print("加载完成！")
    else:
        print("未找到缓存，开始构建索引（首次运行耗时较长）...")
        # 如果 CSV 不存在则重新下载
        if not os.path.exists(DATA_PATH):
            import akshare as ak
            os.makedirs("data/AKSHARE_cjzc", exist_ok=True)
            df = ak.stock_info_cjzc_em()
            df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')
            print("CSV 数据下载完成")

        # 加载并切分文档
        loader = CSVLoader(file_path=DATA_PATH, encoding='utf-8-sig')
        documents = loader.load()
        csv_chunks = []
        for doc in documents:
            csv_chunks.extend(split_text(doc.page_content))
        print(f"文本块数量：{len(csv_chunks)}")

        # 构建 Embedding 和 FAISS 索引
        embeddings = HuggingFaceEndpointEmbeddings(
            model="intfloat/multilingual-e5-large",
            task="feature-extraction",
            huggingfacehub_api_token=os.environ.get("HUGGINGFACEHUB_API_TOKEN"),
            )
        faiss_index = FAISS.from_texts(csv_chunks, embeddings)
        faiss_index.save_local(INDEX_PATH)
        with open(CHUNKS_PKL, "wb") as f:
            pickle.dump(csv_chunks, f)
        print("索引构建并保存完成！")

    # 初始化 LLM（使用 HuggingFaceEndpoint）
    llm = HuggingFaceEndpoint(
        repo_id="deepseek-ai/DeepSeek-V4-Pro", # 可替换为其他可用模型，原 deepseek 模型可能不存在
        temperature=0.7,
        max_new_tokens=1024,
    )
    chat = ChatHuggingFace(llm=llm,verbose=True)
    return embeddings, faiss_index, csv_chunks, chat

# 执行初始化（整个过程只需一次）
embeddings_model, vector_store, all_chunks, llm_model = initialize_rag()

# ------------------------------
# 3. 检索函数（与 notebook 一致）
# ------------------------------
def retrieval(query, k=3):
    query_embedding = embeddings_model.embed_query(query)
    query_embedding_np = np.array(query_embedding).astype("float32").reshape(1, -1)
    D, I = vector_store.index.search(query_embedding_np, k=k)
    return [all_chunks[i] for i in I[0]]

# ------------------------------
# 4. 构建增强提示并调用 LLM
# ------------------------------
def augment_prompt(query: str):
    retrieved = retrieval(query)
    source_knowledge = "\n\n".join(retrieved)
    augmented_prompt = f"""使用以下新闻上下文回答问题。如果上下文信息不足，可以结合你自己的知识给出一般性回答。

上下文：
{source_knowledge}

问题：{query}
回答："""
    return augmented_prompt, retrieved

def answer_query(query):
    if not query.strip():
        return "请输入问题。", ""
    prompt, retrieved_docs = augment_prompt(query)
    try:
        response = llm_model.invoke(prompt)
        answer = response.content
        # 格式化检索到的文档，便于在界面显示
        docs_display = "\n\n---\n\n".join(retrieved_docs)
        return answer, docs_display
    except Exception as e:
        return f"调用模型出错：{str(e)}", ""

# ------------------------------
# 5. Gradio 界面
# ------------------------------
with gr.Blocks(title="财经新闻 RAG 问答系统") as demo:
    gr.Markdown("# 📰 财经早餐 RAG 问答系统")
    gr.Markdown("基于东方财富财经早餐新闻构建的知识库，你可以询问 2024-2026 年间的财经政策、市场动态、国际事件等。")

    with gr.Row():
        with gr.Column(scale=4):
            question = gr.Textbox(label="你的问题", placeholder="例如：六月发生了什么重要新闻？", lines=2)
            submit_btn = gr.Button("提问", variant="primary")
        with gr.Column(scale=6):
            answer = gr.Textbox(label="🤖 回答", lines=6, interactive=False)
            with gr.Accordion("📌 相关新闻片段", open=True):
                retrieved_show = gr.Textbox(label="检索到的上下文", lines=8, interactive=False)

    submit_btn.click(fn=answer_query, inputs=question, outputs=[answer, retrieved_show])

    gr.Examples(
        examples=[
            "六月有什么重要的财经新闻？",
            "关于人工智能的政策有哪些？",
            "美伊冲突对市场有什么影响？",
            "哪些公司发布了 IPO 相关消息？",
        ],
        inputs=question
    )

# 启动服务，share=True 会生成公网链接（有效期72小时）
demo.launch(share=True, debug=True)

正在初始化 RAG 系统...
发现已有 FAISS 索引和文本块缓存，直接加载...
加载完成！
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b2cf3b20c7e1a706ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b2cf3b20c7e1a706ab.gradio.live
